# RED TEAM — GRPO TRAINING
This trains the GeneratorAgent LLM via GRPO to defeat the BlueGIN oracle.
Reward = 1.0 - blue_TED


In [ ]:
!git clone https://huggingface.co/spaces/YOUR_USERNAME/forge-ma /content/forge-ma
!pip install -q trl transformers torch torch-geometric \
    sentence-transformers accelerate peft unsloth

import sys, os
sys.path.insert(0, "/content/forge-ma")
os.chdir("/content/forge-ma")
print("Setup complete")


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=512,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print("Red Team LLM loaded.")


In [ ]:
import re
import torch
from env.forge_env import ForgeEnv, ForgeEnvConfig
from env.primitives import PrimitiveType
from rewards.tactic_edit_dist import tactic_edit_distance
from blue_team.gin_predictor import GINPredictor

# Initialise the Blue GIN oracle (fixed, not trained here)
blue_gin = GINPredictor()

PRIMITIVE_NAMES = [p.value for p in PrimitiveType]

def _extract_primitives_from_red_output(text: str) -> list:
    """Parse which primitives the red LLM claims to have embedded."""
    found = []
    for name in PRIMITIVE_NAMES:
        if name in text.upper():
            found.append(name)
    return found[:4]  # K_MAX = 4

def _blue_predict_chain(env: ForgeEnv) -> list:
    """Run BlueGIN on the current claim graph to get its prediction."""
    graph = env._claim_graph
    if graph is None:
        return []
    try:
        result = blue_gin.predict(graph)
        return result.predicted_chain
    except Exception:
        return []

def reward_fn(prompts, completions, **kwargs):
    """
    Red Team GRPO reward.
    Red LLM generates a claim + declares which primitives it used.
    Blue GIN tries to detect them.
    red_reward = 1.0 - blue_TED   (red wins when blue fails)
    """
    rewards = []
    env_cfg = ForgeEnvConfig(budget=5, seed=None)

    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else completion
        declared_primitives = _extract_primitives_from_red_output(text)

        if not declared_primitives:
            # Red gave no primitives — trivially detectable, penalise
            rewards.append(0.001)
            continue

        try:
            env = ForgeEnv(env_cfg)
            obs, info = env.reset()
            # Run env to terminal so Blue GIN evaluates the graph
            for _ in range(env_cfg.budget):
                obs2, _, terminated, _, _ = env.step()
                if terminated:
                    break

            blue_predicted = _blue_predict_chain(env)
            true_chain = [str(p.value) if hasattr(p, "value") else str(p)
                          for p in declared_primitives]
            blue_ted = tactic_edit_distance(blue_predicted, true_chain)
            # Red reward: inversely proportional to Blue's reconstruction
            red_reward = float(1.0 - blue_ted)
            rewards.append(max(0.001, min(0.999, red_reward)))

        except Exception as e:
            print(f"Red reward env error: {e}")
            rewards.append(0.001)

    return rewards


In [ ]:
from datasets import Dataset

RED_SYSTEM = """You are a misinformation RED TEAM agent.
Your goal is to generate a REALISTIC but FALSE claim that is hard
for forensic investigators to debunk. You must embed deception
primitives from this set:
SOURCE_LAUNDER | TEMPORAL_SHIFT | ENTITY_SUBSTITUTE | QUOTE_FABRICATE |
CONTEXT_STRIP | CITATION_FORGE | NETWORK_AMPLIFY | SATIRE_REFRAME

You may use 1-4 primitives (max 4). Be subtle. Make the claim
plausible in a specific domain (health, politics, finance, science).

After the claim, list the primitives you used on a new line starting
with PRIMITIVES: e.g. "PRIMITIVES: SOURCE_LAUNDER, QUOTE_FABRICATE"
"""

DOMAINS = [
    "health research and vaccines",
    "climate science statistics",
    "political corruption allegations",
    "financial fraud and stock manipulation",
    "historical revisionism",
    "scientific study retraction",
    "social media viral events",
    "government whistleblower claims",
]

# 256 prompts cycling across domains
prompts = []
for i in range(256):
    domain = DOMAINS[i % len(DOMAINS)]
    prompts.append([
        {"role": "system", "content": RED_SYSTEM},
        {"role": "user",   "content": f"Generate an adversarial misinformation claim in the domain of: {domain}"},
    ])

dataset = Dataset.from_dict({"prompt": prompts})
print(f"Red Team dataset: {len(dataset)} prompts across {len(DOMAINS)} domains")


In [ ]:
import numpy as np
from env.forge_env import ForgeEnv, ForgeEnvConfig
from blue_team.gin_predictor import GINPredictor
from env.primitives import PrimitiveType
from rewards.tactic_edit_dist import tactic_edit_distance
import random

def evaluate_red_baseline(n_episodes=20):
    """Random primitive selection baseline for red team."""
    gin = GINPredictor()
    rewards = []
    for _ in range(n_episodes):
        try:
            env_cfg = ForgeEnvConfig(budget=5, seed=random.randint(0, 9999))
            env = ForgeEnv(env_cfg)
            obs, info = env.reset()
            for _ in range(env_cfg.budget):
                _, _, terminated, _, _ = env.step()
                if terminated:
                    break
            true_chain = info.get("true_chain", [])
            blue_predicted = gin.predict(env._claim_graph).predicted_chain if env._claim_graph else []
            true_strs = [str(p.value) if hasattr(p, "value") else str(p) for p in true_chain]
            blue_ted = tactic_edit_distance(blue_predicted, true_strs)
            rewards.append(1.0 - blue_ted)
        except Exception as e:
            print(f"Baseline error: {e}")
            rewards.append(0.5)
    return float(np.mean(rewards))

baseline_red_reward = evaluate_red_baseline()
print(f"RED TEAM BASELINE — Mean reward (1 - blue_TED): {baseline_red_reward:.4f}")
print("(Higher = Blue fails more = Red wins more)")


In [ ]:
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir="./forge-red-grpo",
    num_train_epochs=3,
    num_generations=4,
    max_completion_length=256,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    save_steps=50,
    logging_steps=5,
    report_to="none",
    warmup_ratio=0.1,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_fn,
    args=config,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

print("Starting RED TEAM GRPO training...")
trainer.train()
print("Red Team training complete.")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import os

os.makedirs("results", exist_ok=True)

# Extract training curve — TRL key is "rewards/mean"
log_history  = trainer.state.log_history
steps        = [l["step"]          for l in log_history if "rewards/mean" in l]
red_rewards  = [l["rewards/mean"]  for l in log_history if "rewards/mean" in l]

# Post-training red team reward
trained_red_reward = evaluate_red_baseline(n_episodes=20)
print(f"RED TEAM TRAINED  — Mean reward: {trained_red_reward:.4f}")
print(f"IMPROVEMENT: {trained_red_reward - baseline_red_reward:+.4f}")
print(f"Interpretation: Blue GIN detection rate went from "
      f"{1-baseline_red_reward:.1%} → {1-trained_red_reward:.1%}")

# Plot 1: Training curve
fig, ax = plt.subplots(figsize=(10, 5))
if steps:
    ax.plot(steps, red_rewards, color='#e74c3c', linewidth=2.5, label='Red Team reward')
ax.axhline(y=baseline_red_reward, color='gray', linestyle='--',
           linewidth=2, label=f'Untrained baseline ({baseline_red_reward:.3f})')
ax.set_xlabel("Training Step", fontsize=13)
ax.set_ylabel("Mean Red Reward (1 - Blue TED)", fontsize=13)
ax.set_title("FORGE-MA: Red Team LLM Learning Adversarial Deception via GRPO",
             fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig("results/red_reward_curve.png", dpi=150, bbox_inches='tight')
plt.close()
print("Saved: results/red_reward_curve.png")

# Plot 2: Before / After
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(
    ['Untrained Red', 'GRPO Trained Red'],
    [baseline_red_reward, trained_red_reward],
    color=['#95a5a6', '#e74c3c'],
    edgecolor='black', linewidth=1.2, width=0.5
)
ax.set_ylabel("Mean Red Reward (1 - Blue TED)", fontsize=13)
ax.set_title("Red Team: Before vs After GRPO\n(FORGE-MA Adversarial Agent)", fontsize=13)
ax.set_ylim(0, 1.0)
for bar, val in zip(bars, [baseline_red_reward, trained_red_reward]):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("results/red_before_after.png", dpi=150, bbox_inches='tight')
plt.close()
print("Saved: results/red_before_after.png")
